# 1. Ingest and parse PDFs

This notebook downloads sample PDFs (e.g. from [dbdemos-dataset/databricks-pdf-documentation](https://github.com/databricks-demos/dbdemos-dataset/tree/main/llm/databricks-pdf-documentation)), parses them into text, and writes one row per document to a Delta table.

**Output:** A Delta table with columns `content`, `parser_status`, `doc_uri`, `last_modified`.

**Next:** Run notebook `02-chunk-index.ipynb` to chunk this table and build a Vector Search index.

## Config

Set your Unity Catalog catalog, schema, volume (for source PDFs), and the name of the parsed-docs Delta table.

In [0]:
CATALOG = "main"
SCHEMA = "default"
VOLUME_NAME = "pdf_docs"
PARSED_TABLE = f"{CATALOG}.{SCHEMA}.rag_parsed_docs"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
print(f"Source: {VOLUME_PATH}")
print(f"Output table: {PARSED_TABLE}")

## Ensure volume exists and download sample PDFs

If the volume is empty, download PDFs from the Databricks demo dataset.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import VolumeType
import requests
import os
import tempfile

w = WorkspaceClient()
try:
    w.volumes.get(full_name=f"{CATALOG}.{SCHEMA}.{VOLUME_NAME}")
except Exception:
    w.volumes.create(volume_name=VOLUME_NAME, catalog_name=CATALOG, schema_name=SCHEMA, volume_type=VolumeType.MANAGED)

files = dbutils.fs.ls(VOLUME_PATH)
if not files:
    def download_pdfs(volume_path, owner="databricks-demos", repo="dbdemos-dataset", path="/llm/databricks-pdf-documentation"):
        r = requests.get(f"https://api.github.com/repos/{owner}/{repo}/contents{path}")
        r.raise_for_status()
        with tempfile.TemporaryDirectory() as tmp:
            for f in r.json():
                if not f.get("download_url") or "NOTICE" in f.get("name", ""):
                    continue
                content = requests.get(f["download_url"]).content
                local = os.path.join(tmp, f["name"])
                with open(local, "wb") as out:
                    out.write(content)
                dbutils.fs.cp(f"file:{local}", f"{volume_path}/{f['name']}", overwrite=True)
                print(f"Downloaded {f['name']}")
    download_pdfs(VOLUME_PATH)
else:
    print(f"Volume already has {len(files)} item(s). Skipping download.")

## Parse PDFs and write to Delta

Uses PyMuPDF to extract text as markdown. Install `pymupdf4llm` and `fitz` (PyMuPDF) if not present.

In [0]:
%pip install pymupdf4llm PyMuPDF --quiet
dbutils.library.restartPython()

In [0]:
import fitz
import pymupdf4llm
from urllib.parse import urlparse
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import pyspark.sql.functions as F

def parse_pdf(content: bytes, path: str, modification_time, doc_length: int) -> tuple:
    """Parse PDF to markdown. Returns (content, parser_status, doc_uri, last_modified)."""
    try:
        doc = fitz.Document(stream=content, filetype="pdf")
        md_text = pymupdf4llm.to_markdown(doc)
        doc_uri = urlparse(path).path or path
        return (md_text.strip(), "SUCCESS", doc_uri, modification_time)
    except Exception as e:
        return ("", f"ERROR: {e}", path, modification_time)

schema = StructType([
    StructField("content", StringType()),
    StructField("parser_status", StringType()),
    StructField("doc_uri", StringType()),
    StructField("last_modified", TimestampType()),
])

parse_udf = F.udf(parse_pdf, schema)

raw_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(VOLUME_PATH)
)
raw_df = raw_df.filter(F.col("path").rlike("\.pdf$"))
if raw_df.count() == 0:
    raise ValueError(f"No PDF files found under {VOLUME_PATH}")

parsed_df = raw_df.withColumn(
    "parsed",
    parse_udf("content", "path", "modificationTime", "length")
).select(
    F.col("parsed.content").alias("content"),
    F.col("parsed.parser_status").alias("parser_status"),
    F.col("parsed.doc_uri").alias("doc_uri"),
    F.col("parsed.last_modified").alias("last_modified"),
)

parsed_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(PARSED_TABLE)
print(f"Wrote {parsed_df.count()} rows to {PARSED_TABLE}")
display(spark.table(PARSED_TABLE))